## Rule-based data integration pipeline (Sorted Neighbourhood Blocker)

In [2]:
from pathlib import Path

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"

In [3]:
from PyDI.io import load_parquet

amazon_books = load_parquet(
    DATA_DIR / "amazon_new.parquet",
    name="amazon_books"
)

goodreads = load_parquet(
    DATA_DIR / "goodreads_new.parquet",
    name="goodreads"
)

metabooks = load_parquet(
  DATA_DIR / "metabooks_new.parquet",
  name="metabooks"
)

In [4]:
from PyDI.entitymatching import SortedNeighbourhoodBlocker

sn_blocker_m2a = SortedNeighbourhoodBlocker(
    metabooks, amazon_books,
    key='title_norm',
    window=20,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)


sn_blocker_m2g = SortedNeighbourhoodBlocker(
    metabooks, goodreads,
    key='title_norm',
    window=20,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

/Users/abd/Developer/team_project_data_integration/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from PyDI.entitymatching import StringComparator, NumericComparator

comparators_m2a = [
    StringComparator(column="title_norm", similarity_function="cosine"),
    StringComparator(column="author_norm", similarity_function="cosine", preprocess=str.lower),
    StringComparator(column="publisher", similarity_function="cosine", preprocess=str.lower),
    StringComparator(column="isbn_clean", similarity_function="cosine"),
    NumericComparator(column="publish_year", max_difference=1),
]

comparators_m2g = comparators_m2a + [
    StringComparator(column="genres", similarity_function="jaccard",
                     preprocess=str.lower, list_strategy="concatenate"),
    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
    NumericComparator(column="rating", max_difference=0.2),
]

In [6]:
from PyDI.entitymatching import RuleBasedMatcher

matcher = RuleBasedMatcher()

# matching metabooks > amazon
sn_corr_m2a = matcher.match(
    df_left=metabooks,
    df_right=amazon_books, 
    candidates=sn_blocker_m2a,
    comparators=comparators_m2a,
    weights=[0.4, 0.2, 0.1, 0.2, 0.1], 
    threshold=0.7,
    id_column='id'
)

# matching metabooks > goodreads
sn_corr_m2g = matcher.match(
    df_left=metabooks,
    df_right=goodreads, 
    candidates=sn_blocker_m2g,
    comparators=comparators_m2g,

    weights=[0.4, 0.25, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05], 
    threshold=0.7,
    id_column='id'
)

In [7]:
# data fusion for sorted neighbourhood blocker
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust
import pandas as pd

amazon_books.attrs["trust_score"] = 3
metabooks.attrs["trust_score"] = 2
goodreads.attrs["trust_score"] = 1

all_correspondences = pd.concat([sn_corr_m2a, sn_corr_m2g], ignore_index=True)

len(all_correspondences)

30754

In [8]:
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title_norm', longest_string)
strategy.add_attribute_fuser('author_norm', longest_string)
strategy.add_attribute_fuser('publish_year', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publisher', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('isbn_clean', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('price', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('rating', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres', union)

In [9]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_standard_blocker.jsonl")

fused_sn_blocker = engine.run(
    datasets=[amazon_books, metabooks, goodreads],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)

print(f'Fused rows: {len(fused_sn_blocker):,}')
display(fused_sn_blocker.head())

Fused rows: 27,618


,_id,_fusion_group_id,_fusion_sources,author_norm,price,title,publisher,publish_year,title_norm,author,...,id,rating_number,language,rating,isbn_clean,page_count,_fusion_confidence,_fusion_metadata,book_format,edition
0,amazon_25392,group_0,"[metabooks, amazon_books]",anne rice,2.890000,Servant of the Bones,Random House Audio,1996.0,servant of the bones a novel,Anne Rice,...,amazon_25392,570,English,4.6,0679438335,NaN,0.703297,"{'author_norm_rule': 'longest_string', 'author...",NaN,NaN
1,metabooks_257118,group_1,"[goodreads, metabooks]",santa montefiore,4.650000,The Woman from Paris,Simon & Schuster,2013.0,the woman from paris,Santa Montefiore,...,metabooks_257118,260,English,4.1,1451676689,400.0,0.733333,"{'author_norm_rule': 'longest_string', 'author...",Hardcover,nan
2,amazon_249912,group_2,"[metabooks, amazon_books]",david a adler,30.389999,A Picture Book of Jesse Owens (Picture Book Bi...,Holiday House,1992.0,a picture book of jesse owens picture book bio...,David A. Adler,...,amazon_249912,38,English,4.8,082340966X,32.0,0.653846,"{'author_norm_rule': 'longest_string', 'author...",NaN,NaN
3,metabooks_310763,group_3,"[goodreads, metabooks]",a h holt,5.980000,Blanco Sol,AmazonEncore,2005.0,blanco sol,A. H. Holt,...,metabooks_310763,159,English,4.2,080349730X,182.0,0.737500,"{'author_norm_rule': 'longest_string', 'author...",Hardcover,nan
4,metabooks_152845,group_4,"[amazon_books, metabooks]",pete dexter,5.350000,Brotherly Love,Penguin Books,1992.0,brotherly love,Pete Dexter,...,metabooks_152845,77,English,4.3,0140167730,288.0,0.769231,"{'author_norm_rule': 'longest_string', 'author...",NaN,NaN


In [11]:
fused_sn_blocker.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27618 entries, 0 to 27617
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   _id                 27618 non-null  object 
 1   _fusion_group_id    27618 non-null  object 
 2   _fusion_sources     27618 non-null  object 
 3   author_norm         27618 non-null  object 
 4   price               25429 non-null  float64
 5   title               27618 non-null  object 
 6   publisher           27618 non-null  object 
 7   publish_year        27611 non-null  float64
 8   title_norm          27618 non-null  object 
 9   author              27618 non-null  object 
 10  genres              26402 non-null  object 
 11  id                  27618 non-null  object 
 12  rating_number       27618 non-null  int64  
 13  language            27496 non-null  object 
 14  rating              27618 non-null  float64
 15  isbn_clean          27618 non-null  object 
 16  page

In [ ]:
fused_sn_blocker.to_parquet(PIPELINE_DIR / "fused_standard_blocker.parquet")